# 5.3 · Implementación II: subagente, trazas y primera ejecución completa

**Hito:** el agente procesa los cinco correos de la bandeja y deja trazas y un JSON por correo.

## 1. Preparación

Además de lo del 5.2: `buscar_kb` (reto del 5.2) y, del 4.4, el hook `auditar` con sus utilidades para leer y borrar el log.

In [ ]:
%load_ext autoreload
%autoreload 2

from dotenv import load_dotenv
load_dotenv()

import sys; sys.path.append("../..")   # para importar desde la raíz del taller
from pathlib import Path
from config import MODEL

import json
import herramientas
from claude_agent_sdk import (
    query, tool, create_sdk_mcp_server, ClaudeAgentOptions, AgentDefinition, HookMatcher,
    ResultMessage, ToolAnnotations,
)
from taller_utils import imprimir, buscar_kb, auditar, CONTEXTO_LOG, leer_log, borrar_log, restaurar_datos

CORREOS = herramientas.listar_bandeja()[:5]   # el 006 es para el bloque 6
CORREOS

## 2. Herramientas

Las del 5.2, sin cambios.

In [ ]:
def texto(dato):
    if not isinstance(dato, str):
        dato = json.dumps(dato, ensure_ascii=False)
    return {"content": [{"type": "text", "text": dato}]}

def error(mensaje):
    return {"content": [{"type": "text", "text": mensaje}], "is_error": True}

@tool("leer_correo",
      "Lee un correo de la bandeja de soporte a partir de su nombre de fichero "
      "(p. ej. '001_exportar_excel.txt'). Devuelve el texto completo: remitente, asunto y cuerpo.",
      {"nombre": str},
      annotations=ToolAnnotations(readOnlyHint=True))
async def leer_correo(args):
    try:
        return texto(herramientas.leer_correo(args["nombre"]))
    except FileNotFoundError as e:
        return error(str(e))

@tool("buscar_incidencias",
      "Busca incidencias de soporte existentes. Úsala antes de crear cualquier incidencia, para "
      "comprobar si ya hay una abierta sobre el mismo problema. Todos los filtros son opcionales y se "
      "combinan. Devuelve una lista JSON con id, cliente, asunto, estado, prioridad y fecha; [] si no "
      "hay coincidencias.",
      {
          "type": "object",
          "properties": {
              "texto": {"type": "string", "description": "Palabras clave a buscar en asunto y descripción"},
              "estado": {"type": "string", "enum": ["abierta", "cerrada"]},
              "cliente": {"type": "string", "description": "Nombre comercial corto del cliente, p. ej. 'ACME'"},
          },
      },
      annotations=ToolAnnotations(readOnlyHint=True))
async def buscar_incidencias(args):
    return texto(herramientas.buscar_incidencias(**args))

@tool("crear_incidencia",
      "Crea una incidencia nueva y devuelve la incidencia creada con su id. Úsala solo cuando el "
      "correo describe un problema y no existe ya una incidencia abierta sobre él para ese cliente.",
      {
          "type": "object",
          "properties": {
              "cliente": {"type": "string", "description": "Nombre comercial corto del cliente"},
              "asunto": {"type": "string", "description": "Resumen en menos de 10 palabras"},
              "descripcion": {"type": "string", "description": "Qué ocurre, desde cuándo y a quién afecta"},
              "prioridad": {"type": "string", "enum": ["alta", "media", "baja"]},
          },
          "required": ["cliente", "asunto", "descripcion", "prioridad"],
      })
async def crear_incidencia(args):
    return texto(herramientas.crear_incidencia(**args))

## 3. Subagente clasificador

El del 4.3, ajustado al triaje: en lugar de buscar los criterios en la base de conocimiento, se los pasamos **en su prompt**. Así no necesita ninguna herramienta (`tools=[]`): mínimo privilegio.

In [ ]:
CRITERIOS = Path("../../datos/kb/prioridades.md").read_text(encoding="utf-8")

clasificador = AgentDefinition(
    description=("Especialista en clasificar incidencias de soporte. Úsalo cuando haya que "
                 "decidir la prioridad de una incidencia nueva."),
    prompt=("Eres un clasificador de incidencias. Dada la descripción de un problema, aplica estos "
            "criterios y devuelve SOLO: prioridad (alta/media/baja), categoría "
            "(facturación/técnico/cuenta/otro) y una justificación de una línea.\n\n" + CRITERIOS),
    tools=[],   # no necesita herramientas: los criterios van en su prompt
    model="claude-haiku-5-5",
)

## 4. Hook de auditoría

`auditar` es el del 4.4, con una mejora: añade a cada línea del log lo que haya en `CONTEXTO_LOG`. Antes de triar cada correo pondremos ahí su nombre, para poder filtrar las trazas por correo.

## 5. Salida estructurada

El JSON de la pizarra, como modelo de Pydantic:

In [ ]:
from pydantic import BaseModel
from typing import Literal

class Triaje(BaseModel):
    correo: str
    es_incidencia: bool
    cliente: str | None
    incidencia_id: str | None
    accion: Literal["creada", "existente", "ninguna"]
    prioridad: Literal["alta", "media", "baja"] | None
    articulo_kb: str | None
    respuesta_propuesta: str

En el SDK, el formato se pasa con `output_format` (un JSON Schema, que Pydantic genera con `model_json_schema()`). El resultado llega validado en `ResultMessage.structured_output`.

El system prompt gana una línea en el paso 3 (el clasificador) y las opciones, el subagente, los hooks y el formato de salida:

In [ ]:
SYSTEM = """Eres un asistente de triaje de soporte. Para el correo indicado:
1. Determina si describe un problema real. Si no, no crees nada.
2. Identifica al cliente y busca si ya existe una incidencia abierta sobre el mismo problema.
3. Si no existe, pide la prioridad al subagente clasificador y crea la incidencia.
4. Busca en la base de conocimiento un artículo que ayude y propón una respuesta basada SOLO en él.
Nunca envías nada: tu salida la revisa una persona."""

In [ ]:
soporte = create_sdk_mcp_server(name="soporte", version="1.0.0",
                                tools=[leer_correo, buscar_incidencias, crear_incidencia, buscar_kb])

options = ClaudeAgentOptions(
    model=MODEL,
    system_prompt=SYSTEM,
    mcp_servers={"soporte": soporte},
    tools=["Task"],
    allowed_tools=["Task", "mcp__soporte__leer_correo", "mcp__soporte__buscar_incidencias",
                   "mcp__soporte__crear_incidencia", "mcp__soporte__buscar_kb"],
    agents={"clasificador": clasificador},
    hooks={
        "PreToolUse":  [HookMatcher(hooks=[auditar])],
        "PostToolUse": [HookMatcher(hooks=[auditar])],
    },
    output_format={"type": "json_schema", "schema": Triaje.model_json_schema()},
    env={"CLAUDE_CODE_DISABLE_BACKGROUND_TASKS": "1"},   # que espere al subagente (4.3)
    max_turns=15,
    setting_sources=[],
)

`triar()` ahora devuelve el `ResultMessage`. Antes de empezar, pone el nombre del correo en `CONTEXTO_LOG`.

In [ ]:
async def triar(nombre_correo, **etiquetas):
    CONTEXTO_LOG.clear()
    CONTEXTO_LOG.update(correo=nombre_correo, **etiquetas)   # cada línea del log lleva el correo
    async for msg in query(prompt=f"Tría el correo {nombre_correo}", options=options):
        imprimir(msg)
        if isinstance(msg, ResultMessage):
            return msg

In [ ]:
restaurar_datos()
borrar_log()

r = await triar("003_urgente.txt")
r.structured_output

En las trazas aparece una herramienta `StructuredOutput` que no hemos definido: es el mecanismo del SDK para devolver el JSON. La filtraremos al mostrar las trazas.

## 6. Ejecución completa

Los cinco correos, desde un estado conocido:

In [ ]:
restaurar_datos()
borrar_log()

resultados, coste_total = [], 0
for correo in CORREOS:
    print(f"\n━━━ {correo}")
    r = await triar(correo)
    coste_total += r.total_cost_usd
    if r.subtype == "success" and r.structured_output:
        resultados.append({**r.structured_output, "correo": correo})

La bandeja triada:

In [ ]:
import pandas as pd

df = pd.DataFrame(resultados)
Path("../../logs/resultados.json").write_text(json.dumps(resultados, ensure_ascii=False, indent=2))
print(f"Coste total: ${coste_total:.4f}")
df[["correo", "es_incidencia", "cliente", "accion", "incidencia_id", "prioridad", "articulo_kb"]]

Y las trazas, filtradas por correo (sin `StructuredOutput`):

In [ ]:
def trazas(correo, **filtros):
    log = leer_log()
    log = log[(log.correo == correo) & (log.evento == "PreToolUse") & (log.herramienta != "StructuredOutput")]
    for campo, valor in filtros.items():
        log = log[log[campo] == valor]
    return log[["herramienta", "entrada"]]

In [ ]:
trazas("002_duplicado.txt")

### Los criterios de éxito del 5.1

In [ ]:
def revisar_criterios(df):
    log = leer_log()
    crea = lambda correo: ((log.correo == correo) & (log.evento == "PreToolUse")
                           & (log.herramienta == "mcp__soporte__crear_incidencia")).any()
    fila = df.set_index("correo").loc
    criterios = [
        ("duplicado → existente, sin crear",
         fila["002_duplicado.txt", "accion"] == "existente" and not crea("002_duplicado.txt")),
        ("urgente → prioridad alta",
         fila["003_urgente.txt", "prioridad"] == "alta"),
        ("felicitación → no es incidencia, sin crear",
         not fila["005_felicitacion.txt", "es_incidencia"] and not crea("005_felicitacion.txt")),
    ]
    for nombre, ok in criterios:
        print("✅" if ok else "❌", nombre)

In [ ]:
revisar_criterios(df)

Si alguno falla, lo depuramos en el siguiente vídeo. Si en tu ejecución pasan todos, no te confíes: los agentes no son deterministas, y en el 5.4 partimos de una versión en la que el fallo es seguro.

## 🔸 Reto

Añade al esquema un campo `confianza` (`alta`/`media`/`baja`) y pide en el system prompt que sea baja cuando el cliente no esté claro. Pruébalo con `004_ambiguo.txt`.